# Kronos × 事件合约（BTC/ETH 10min/30min/1h）— Colab

检验 Kronos 预训练模型预测"下一根 K 线涨跌"能否越过事件合约盈亏平衡胜率（10min 赔率 80% → 55.6%；30min/1h 赔率 85% → 54.1%）。
口径见 `event_contract/README.md`。

**准备**：菜单「代码执行程序 → 更改运行时类型」选 **T4 GPU**。预测结果写到 Google Drive，断线后重跑同一格会跳过已完成月份。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BRANCH = 'event-contract'
if not os.path.exists('/content/Kronos'):
    !git clone -q -b {BRANCH} https://github.com/Musenxi/Kronos.git /content/Kronos
%cd /content/Kronos
!pip install -q einops huggingface_hub safetensors tabulate
# 数据与预测放 Drive（1m 数据约 60MB）
!mkdir -p /content/drive/MyDrive/kronos_event/data /content/drive/MyDrive/kronos_event/output
!rm -rf event_contract/data event_contract/output
!ln -s /content/drive/MyDrive/kronos_event/data event_contract/data
!ln -s /content/drive/MyDrive/kronos_event/output event_contract/output

In [ ]:
!python event_contract/run_event.py download

## 滚动预测
先跑 1h 看每月耗时，再跑 30min / 10min。显存不足调小 `--batch`；10min 太慢加 `--stride 2`。

In [ ]:
!python event_contract/run_event.py predict --tf 1h --batch 64

In [ ]:
!python event_contract/run_event.py predict --tf 30min --batch 64

In [ ]:
!python event_contract/run_event.py predict --tf 10min --batch 64

## 评估（发现期选阈值 → 留出期一次性评估）

In [ ]:
!python event_contract/run_event.py evaluate > /dev/null
from IPython.display import Markdown, display
display(Markdown(open('event_contract/output/report.md', encoding='utf-8').read()))

In [ ]:
!cd event_contract && zip -qr /content/kronos_event_results.zip output
from google.colab import files
files.download('/content/kronos_event_results.zip')